# 00 — Stream health

Is the stream alive, and is it complete? Three separate questions, answered
separately: what is in the log, what is arriving now, and what went missing
in between.

**Before running:** you need a broker with data in it.

- Local: `make stream-local` in another terminal (brings up compose Kafka,
  creates the topics, runs the Binance + Coinbase producers on the host).
- MSK: `make up`, then set `FDAI_TARGET=msk` — or call
  `devlab.from_terraform()` below, which reads the endpoint straight from the
  stack outputs and cannot go stale.

In [1]:
import devlab
from devlab import frames, health

target = devlab.resolve()  # $FDAI_TARGET: "local" (default), "msk", or "terraform"
target

Target(name='local', bootstrap='127.0.0.1:9092', sasl_username=None)

## What is in the log

`messages` is `high - low` per partition, summed — what is **currently
retained**, not what was ever written. `md.trades.v1` ages out at 24h by
design, so this number plateaus rather than growing forever.

A topic at 0 is normal for everything except `md.trades.v1`: no other
producer code exists yet (see `docs/ARCHITECTURE.md`).

In [2]:
frames.frame(devlab.topics(target))

,name,partitions,messages
0,_dlq.md.bars.v1,1,0
1,_dlq.md.book.depth.v1,1,0
2,_dlq.md.book.top.v1,1,0
3,_dlq.md.trades.v1,1,0
4,_dlq.news.articles.v1,1,0
5,md.bars.v1,3,0
6,md.book.depth.v1,6,0
7,md.book.top.v1,6,0
8,md.trades.v1,6,6065
9,news.articles.v1,3,0


In [7]:
frames.frame(devlab.topics(target))

,name,partitions,messages
0,_dlq.md.bars.v1,1,0
1,_dlq.md.book.depth.v1,1,0
2,_dlq.md.book.top.v1,1,0
3,_dlq.md.trades.v1,1,0
4,_dlq.news.articles.v1,1,0
5,md.bars.v1,3,0
6,md.book.depth.v1,6,0
7,md.book.top.v1,6,0
8,md.trades.v1,6,38176
9,news.articles.v1,3,0


## Is the keying working

Trades are keyed `venue|venue_symbol` so one instrument on one venue stays
ordered on one partition. With 8 instruments across 2 venues over 6
partitions, expect *uneven but non-empty* partitions. A single partition
holding everything means the key is not being set.

In [3]:
frames.frame(devlab.partitions(target, "md.trades.v1"))

,topic,partition,low,high,messages
0,md.trades.v1,0,0,675,675
1,md.trades.v1,1,0,35,35
2,md.trades.v1,2,0,1609,1609
3,md.trades.v1,3,0,1869,1869
4,md.trades.v1,4,0,1076,1076
5,md.trades.v1,5,0,885,885


## Is anything arriving right now

Reads from `latest`, so this measures live arrivals only. Zero here with a
non-zero count above means the producers stopped — not that the topic is
empty.

In [11]:
report = devlab.rate(target, seconds=10.0)
print(f"{report.messages} trades in {report.seconds:.1f}s = {report.per_second:.1f}/s")
report.by_venue

127 trades in 10.0s = 12.7/s


{'binance': 92, 'coinbase': 35}

In [5]:
frames.frame([{"instrument_id": k, "trades": v} for k, v in report.by_instrument.items()])

,instrument_id,trades
0,ETH-USD,27
1,BTC-USD,25
2,XRP-USD,17
3,SOL-USD,14
4,DOGE-USD,4
5,LINK-USD,3
6,AVAX-USD,1


## Did anything go missing

Replays sequence numbers through `SequenceTracker` — the same detector the
producer runs inline.

Two caveats worth keeping in mind:

- **Coinbase is skipped, not reported clean.** Its `sequence_num` is
  connection-wide while the topic is partitioned by symbol, so any gap found
  here would be an artefact of reading across partitions.
- **A gap here is weaker evidence than one from `IngestRunner`.** It means the
  record never reached Kafka *or* is no longer retained — and the runner may
  already have repaired it via REST, landing it at a later offset.

In [6]:
recent = devlab.collect(target, limit=5_000, seconds=30.0, offset_reset="earliest")
gaps = health.sequence_gaps(recent)

print(f"checked   {gaps.checked} sequenced records")
print(f"gaps      {len(gaps.gaps)} ({gaps.missing} trades missing)")
print(f"skipped   {gaps.skipped_venues or 'none'} (connection-scoped sequence)")

frames.frame(gaps.gaps) if gaps.gaps else "no gaps found"

2026-08-09 21:27:40 [warning  ] gap_detected                   last_seen=712300364 missing=9280 next_seen=712309645 symbol=XRPUSDT venue=binance
2026-08-09 21:27:40 [warning  ] gap_detected                   last_seen=711832444 missing=6231 next_seen=711838676 symbol=DOGEUSDT venue=binance
2026-08-09 21:27:40 [warning  ] gap_detected                   last_seen=4031028122 missing=51295 next_seen=4031079418 symbol=BTCUSDT venue=binance
checked   1399 sequenced records
gaps      3 (66806 trades missing)
skipped   ['coinbase'] (connection-scoped sequence)


,venue,venue_symbol,last_seen,next_seen,missing_count
0,binance,XRPUSDT,712300364,712309645,9280
1,binance,DOGEUSDT,711832444,711838676,6231
2,binance,BTCUSDT,4031028122,4031079418,51295


## Consumer lag

For a named group — e.g. once a Databricks Bronze reader is running. `devlab`'s
own reads use random group ids with auto-commit off, so they never appear here
and never move anyone else's offsets.

`committed = None` means the group has never committed that partition.

In [ ]:
frames.frame(health.lag(target, group="fdai-bronze", topic="md.trades.v1"))